# MuscleMap WB + MedSAM — whole mask prompt

Instead of a bounding box, feeds the MuscleMap binary mask directly as SAM's
dense mask prompt. The model uses the full shape of the existing segmentation
rather than just its bounding box.

MedSAM image embedding is computed **once per slice** and reused for all muscles.

**Kernel:** `dafne_clean`

In [ ]:
import glob
import os
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
from skimage import transform
from dafne.config import GlobalConfig
from dafne.utils.sam_mask_refine import load_sam, determine_device

In [ ]:
DEVICE        = determine_device()
MM_SEGS_DIR   = "MuscleMap_segs"
IMAGE_GLOB    = "myosegmenTUM/*/ImageData/*FATFRACTION/*FATFRACTION_stack*.nii"
OUTPUT_DIR    = "MuscleMap_WB_maskprompt"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Device:", DEVICE)

LABEL_MAP = {
    7101: "Vastus_Lateralis_L",
    7102: "Vastus_Lateralis_R",
    7111: "Vastus_Intermedius_L",
    7112: "Vastus_Intermedius_R",
    7121: "Vastus_Medialis_L",
    7122: "Vastus_Medialis_R",
    7131: "Rectus_Femoris_L",
    7132: "Rectus_Femoris_R",
    7141: "Sartorius_L",
    7142: "Sartorius_R",
    7151: "Gracilis_L",
    7152: "Gracilis_R",
    7161: "Semimembranosus_L",
    7162: "Semimembranosus_R",
    7171: "Semitendinosus_L",
    7172: "Semitendinosus_R",
    7181: "Biceps_Femoris_L",
    7182: "Biceps_Femoris_R",
    7201: "Adductor_Magnus_L",
    7202: "Adductor_Magnus_R",
}
print(f"{len(LABEL_MAP)} muscle labels defined")

In [ ]:
GlobalConfig['SAM_MODEL'] = 'Med Sam'
sam_model = load_sam('Med Sam')
sam_model.eval()
print("MedSAM loaded on", DEVICE)

In [ ]:
def medsam_mask_inference(sam_model, image_embedding, coarse_mask, H, W):
    """
    Refine a coarse binary mask using SAM's dense mask prompt.

    SAM expects the mask at 256x256 (4x the 64x64 image embedding size).
    The mask shape is used as a dense spatial prior; no bounding box is needed.
    """
    mask_256 = transform.resize(
        coarse_mask.astype(float), (256, 256), order=0, preserve_range=True
    )
    mask_tensor = torch.tensor(mask_256[None, None]).float().to(image_embedding.device)

    with torch.no_grad():
        sparse_emb, dense_emb = sam_model.prompt_encoder(
            points=None, boxes=None, masks=mask_tensor
        )
        low_res_logits, _ = sam_model.mask_decoder(
            image_embeddings=image_embedding,
            image_pe=sam_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False,
        )

    pred = F.interpolate(
        torch.sigmoid(low_res_logits), size=(H, W), mode="bilinear", align_corners=False
    )
    return (pred.squeeze().cpu().numpy() > 0.5).astype(np.uint8)

In [ ]:
image_files = sorted(glob.glob(IMAGE_GLOB))
matched, missing = [], []
for nii_path in image_files:
    stem = os.path.splitext(os.path.basename(nii_path))[0]
    seg_path = os.path.join(MM_SEGS_DIR, f"{stem}_dseg.nii.gz")
    if os.path.exists(seg_path):
        matched.append(nii_path)
    else:
        missing.append(stem)
print(f"Found {len(matched)} stacks with MuscleMap seg, {len(missing)} without")

In [ ]:
for nii_path in matched:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_mm_maskprompt.npz")

    if os.path.exists(out_path):
        print(f"Skipping (already done): {out_path}")
        continue

    seg_path = os.path.join(MM_SEGS_DIR, f"{stem}_dseg.nii.gz")
    print(f"\nProcessing: {nii_path}")

    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)
    H, W      = img_array.shape[1], img_array.shape[2]
    print(f"  Shape: {img_array.shape}")

    seg_sitk  = sitk.ReadImage(seg_path)
    seg_array = sitk.GetArrayFromImage(seg_sitk)

    all_masks = {}

    for slice_idx in range(img_array.shape[0]):
        slice_2d  = img_array[slice_idx]
        seg_slice = seg_array[slice_idx]

        # image embedding — computed once per slice
        img_norm   = slice_2d * 255.0 / (slice_2d.max() + 1e-8)
        img_3c     = np.repeat(img_norm[:, :, None], 3, axis=-1)
        img_1024   = transform.resize(
            img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
        ).astype(np.uint8)
        img_1024   = (img_1024 - img_1024.min()) / np.clip(
            img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
        )
        img_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            image_embedding = sam_model.image_encoder(img_tensor)

        # refine each muscle using its MuscleMap mask shape as prompt
        for label_idx, muscle_name in LABEL_MAP.items():
            mask_arr = (seg_slice == label_idx).astype(np.uint8)

            if mask_arr.any():
                refined = medsam_mask_inference(sam_model, image_embedding, mask_arr, H, W)
            else:
                refined = mask_arr

            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = refined

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f"  slice {slice_idx + 1}/{img_array.shape[0]} done")

    np.savez_compressed(out_path, **all_masks)
    print(f"  Saved -> {out_path}")

print("\nAll done.")

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.npz")))
if results:
    sample = np.load(results[0])
    print("Sample:", results[0])
    for name in sample.files:
        arr = sample[name]
        print(f"  {name}: shape={arr.shape}  voxels={arr.sum()}")
else:
    print("No results yet.")